Speech Splitter

In [14]:
# Fed Speech Cleaner - full method from Shah, Paturi & Chava (2023), for a 2-column CSV
#
# Pipeline (paper section 3.1):
#   1. clean boilerplate around each speech (greetings, thanks, disclaimers, footnotes, URLs)
#   2. split speeches into sentences (NLTK if available, regex fallback otherwise)
#   3. KEYWORD MATCH: keep "target" sentences containing a Panel A1/B1 term (Table 1,
#      Gorodnichenko et al. 2021 dictionary)
#   4. sample ~TARGET_N sentences, balanced across speeches (paper: 5 per file -> 994)
#   5. SENTENCE SPLIT: break mixed-tone sentences at but/however/even though/although/while/;
#      keeping the split only if every segment still contains a Table 1 word
#   6. save everything to one Excel file with 3 sheets
#
# Paste this as a single notebook cell.

from pathlib import Path
import math
import re

import pandas as pd

FED_Speeches_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/filtered_speeches/")
Fed_speech_df = pd.read_csv(FED_Speeches_path / "fed_speeches.csv")

TEXT_COL = "text"
DATE_COL = "date"

SEED = 5768        # one of the paper's seeds
TARGET_N = 1000    # size of the final sentence sample
MIN_WORDS = 6      # drop fragments shorter than this (paper's Fed sentences average ~30 words)

# ---------------------------------------------------------- Table 1 dictionary
PANEL_A1 = ["inflation expectation", "interest rate", "bank rate", "fund rate",
            "price", "economic activity", "inflation", "employment"]
PANEL_B1 = ["unemployment", "growth", "exchange rate", "productivity",
            "deficit", "demand", "job market", "monetary policy"]
PANEL_A2 = ["anchor", "cut", "subdue", "decline", "decrease", "reduce", "low",
            "drop", "fall", "fell", "decelerate", "slow", "pause", "pausing",
            "stable", "non-accelerating", "downward", "tighten"]
PANEL_B2 = ["ease", "easing", "rise", "rising", "increase", "expand", "improve",
            "strong", "upward", "raise", "high", "rapid"]

def term_to_regex(term):
    # word boundaries + optional plural, so "fund rate" also matches "federal funds rate"
    return r"\s+".join(re.escape(w) + "s?" for w in term.lower().split())

def build_pattern(terms):
    return re.compile(r"\b(?:" + "|".join(term_to_regex(t) for t in terms) + r")\b")

TARGET_PATTERN     = build_pattern(PANEL_A1 + PANEL_B1)                        # keyword match
ALL_TABLE1_PATTERN = build_pattern(PANEL_A1 + PANEL_B1 + PANEL_A2 + PANEL_B2)  # split validation

def contains_target(s):
    return bool(TARGET_PATTERN.search(s.lower()))

# ---------------------------------------------------------- boilerplate cleaning
FOOTNOTE_SECTION = re.compile(
    r"(?:^|\n|(?<=[.!?])\s)\s*(?:footnotes?|references|bibliography)\b\s*(?::|\n|\d|$)",
    re.IGNORECASE)

BOILERPLATE_PATTERNS = [
    r"\bthank you\b",
    r"\b(?:would like|want|wish|like) to thank\b",
    r"\bmy thanks\b",
    r"\bit(?:'s| is)(?: a| an)?(?: great| real| distinct)? (?:pleasure|honor|honour|privilege|delight)\b",
    r"\b(?:pleased|delighted|honored|honoured|glad|grateful|happy) to (?:be|join|speak|address|participate|have)\b",
    r"\bgood (?:morning|afternoon|evening)\b",
    r"^\s*welcome\b",
    r"\bviews?\b[^.]{0,80}\bmy own\b",
    r"\bmy own\b[^.]{0,80}\bnot necessarily\b",
    r"\bnot necessarily\b[^.]{0,60}\b(?:those of|reflect|represent|shared|the views?)\b",
    r"\bcongratulat",
    r"^\s*(?:figure|table|chart|slide|exhibit|box)\b",
    r"^\s*(?:sources?|notes?)\s*:",
    r"^\s*references\b",
    r"\breturn to text\b",
    r"\bas prepared for delivery\b",
    r"\bbefore i begin\b",
]
BOILERPLATE_RE = re.compile("|".join(BOILERPLATE_PATTERNS), re.IGNORECASE)
ILLEGAL_XLSX = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def clean_speech_text(text):
    if not isinstance(text, str):
        return ""
    t = text
    m = FOOTNOTE_SECTION.search(t)
    if m and m.start() > len(t) * 0.5:
        t = t[:m.start()]
    t = re.sub(r"return to text", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"\[\d+\]", " ", t)
    t = re.sub(r"(?<=[a-zA-Z\)\"'%])\.(\d{1,2})(?=\s|$)", ". ", t)
    t = ILLEGAL_XLSX.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

def keep_sentence(sent):
    if len(sent.split()) < MIN_WORDS:
        return False
    alpha = sum(ch.isalpha() for ch in sent)
    if alpha / max(len(sent), 1) < 0.5:
        return False
    return not BOILERPLATE_RE.search(sent)

# ---------------------------------------------------------- tokenizer (paper uses NLTK)
try:
    import nltk
    for pkg in ["punkt", "punkt_tab"]:
        try:
            nltk.data.find(f"tokenizers/{pkg}")
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass
    from nltk.tokenize import sent_tokenize
    sent_tokenize("Test one. Test two.")
    print("using NLTK sentence tokenizer")
except Exception:
    _SENT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])")
    def sent_tokenize(text):
        return [s for s in _SENT_RE.split(text) if s.strip()]
    print("NLTK unavailable - using regex fallback tokenizer")

# ---------------------------------------------------------- 1-3: clean, tokenize, keyword match
records = []
for idx, row in Fed_speech_df.iterrows():
    cleaned = clean_speech_text(row[TEXT_COL])
    if not cleaned:
        continue
    for s in (x.strip() for x in sent_tokenize(cleaned)):
        if keep_sentence(s) and contains_target(s):
            records.append({"file_id": idx, "date": row[DATE_COL], "sentence": s})

pool = pd.DataFrame(records).drop_duplicates(subset="sentence").reset_index(drop=True)
n_files = pool["file_id"].nunique()
print(f"{len(pool):,} target sentences from {n_files} speeches "
      f"({len(pool)/max(n_files,1):.1f} per speech)")

# ---------------------------------------------------------- 4: sample ~TARGET_N (seeded)
per_file = max(1, math.ceil(TARGET_N / max(n_files, 1)))
sampled = (pool.sample(frac=1, random_state=SEED)   # shuffle
               .groupby("file_id")
               .head(per_file))                     # balanced quota per speech (paper uses 5)
if len(sampled) < TARGET_N and len(pool) > len(sampled):
    remaining = pool.drop(sampled.index)
    sampled = pd.concat([sampled, remaining.sample(n=min(TARGET_N - len(sampled), len(remaining)),
                                                   random_state=SEED)])
if len(sampled) > TARGET_N:
    sampled = sampled.sample(n=TARGET_N, random_state=SEED)
sampled = sampled.sort_values("file_id").reset_index(drop=True)
print(f"sampled {len(sampled)} sentences (quota {per_file}/speech)")
if len(sampled) < TARGET_N:
    print(f"WARNING: only {len(sampled)} target sentences exist in total.")

# ---------------------------------------------------------- 5: sentence splitting (paper 3.1)
SPLIT_RE = re.compile(
    r"\s*(?:;|,?\s*\bbut\b|,?\s*\bhowever\b,?|\beven though\b|\balthough\b|\bwhile\b)\s*",
    re.IGNORECASE)

def split_sentence(sent):
    parts = [p.strip(" ,.;") for p in SPLIT_RE.split(sent)]
    parts = [p for p in parts if p]
    if len(parts) < 2:
        return [sent]
    if all(ALL_TABLE1_PATTERN.search(p.lower()) for p in parts):
        return parts
    return [sent]

rows = []
for _, r in sampled.iterrows():
    parts = split_sentence(r["sentence"])
    for j, p in enumerate(parts):
        rows.append({"date": r["date"], "sentence": p,
                     "was_split": len(parts) > 1, "segment": j})
sampled_split = pd.DataFrame(rows)
print(f"after splitting: {len(sampled)} -> {len(sampled_split)} sentences")

# ---------------------------------------------------------- 6: one Excel file, 3 sheets
xlsx_path = FED_Speeches_path / f"fed_speeches_sentences_{TARGET_N}.xlsx"
try:
    with pd.ExcelWriter(xlsx_path) as w:
        sampled[["date", "sentence"]].to_excel(w, sheet_name="sampled", index=False)
        sampled_split[["date", "sentence", "was_split"]].to_excel(w, sheet_name="sampled_split", index=False)
        pool[["date", "sentence"]].to_excel(w, sheet_name="all_target_sentences", index=False)
except ModuleNotFoundError:
    raise SystemExit("pandas needs openpyxl to write .xlsx - run:  pip install openpyxl")

print("saved:", xlsx_path)
sampled.head(10)

using NLTK sentence tokenizer
81,551 target sentences from 2527 speeches (32.3 per speech)
sampled 1000 sentences (quota 1/speech)
after splitting: 1000 -> 1019 sentences
saved: /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/filtered_speeches/fed_speeches_sentences_1000.xlsx


,file_id,date,sentence
0,2,1997-12-15 00:00:00,That is just for payment systems; bank supervi...
1,3,1997-12-02 00:00:00,"In addition, in some cases, the glut of semico..."
2,4,1997-11-17 00:00:00,While less flexible labor markets and the abse...
3,5,1997-11-07 00:00:00,Defining such a constant-quality unit of outpu...
4,12,1997-10-14 00:00:00,"More generally, at the root of both the rise i..."
5,14,1997-10-14 00:00:00,The turmoil in the European Exchange Rate mech...
6,15,1997-10-08 00:00:00,"Similarly, more labor-saving equipment can per..."
7,16,1997-10-08 00:00:00,"In this regard, the Basle Committee on Banking..."
8,18,1997-10-01 00:00:00,"For example, in the case of a company that act..."
9,22,1997-09-17 00:00:00,The resulting absence of excess demand in the ...
